# Reefer Challenge Starter Notebook

This notebook builds a very simple valid public leaderboard submission for the hackathon challenge.

What it does:
- reads the released public `target_timestamps.csv`
- aggregates hourly reefer power consumption from `reefer_release.zip`
- uses a naive `t-24h` forecast
- adds a simple `pred_p90_kw = 1.10 * pred_power_kw`
- writes `predictions.csv` for the public leaderboard target hours

This is only a starter. Teams should improve it with better feature engineering, validation, and uncertainty handling.

In [1]:
from __future__ import annotations

import csv
import io
import zipfile
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

def find_public_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "challenge" / "release" / "v1" / "public",
        Path.cwd() / "challenge" / "bundle" / "v1" / "participant_package",
        Path.cwd().parent,
        Path.cwd().parent / "public",
    ]
    for candidate in candidates:
        if (candidate / "reefer_release.zip").exists() and (candidate / "target_timestamps.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the participant data directory.")


PUBLIC_DIR = find_public_dir()
REEFER_ZIP = PUBLIC_DIR / "reefer_release.zip"
TARGETS_CSV = PUBLIC_DIR / "target_timestamps.csv"
OUTPUT_DIR = PUBLIC_DIR / "starter" / "output" if (PUBLIC_DIR / "starter").exists() else PUBLIC_DIR / "output"
OUTPUT_CSV = OUTPUT_DIR / "predictions.csv"

PUBLIC_DIR, REEFER_ZIP.exists(), TARGETS_CSV.exists()

(PosixPath('/Users/wegel/Documents/Projects/Hackathon 2026/participant_package'),
 True,
 True)

In [2]:
def parse_decimal(value: str) -> float | None:
    text = value.strip()
    if not text:
        return None
    try:
        return float(text.replace(",", "."))
    except ValueError:
        return None


def parse_timestamp(value: str) -> datetime:
    text = value.strip()
    if text.endswith("Z"):
        text = text[:-1]
    return datetime.fromisoformat(text.replace(" ", "T"))


def to_hour(ts: datetime) -> datetime:
    return ts.replace(minute=0, second=0, microsecond=0)


def iso_utc(ts: datetime) -> str:
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


In [3]:
def load_target_hours(path: Path) -> list[datetime]:
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        return [parse_timestamp(row["timestamp_utc"]) for row in reader if row.get("timestamp_utc")]


def aggregate_hourly_load_from_release(reefer_zip: Path) -> dict[datetime, float]:
    hourly_power_w = defaultdict(float)
    with zipfile.ZipFile(reefer_zip) as zf:
        with zf.open("reefer_release.csv") as raw:
            text = io.TextIOWrapper(raw, encoding="utf-8-sig", errors="replace", newline="")
            reader = csv.DictReader(text, delimiter=";")
            for row in reader:
                ts = parse_timestamp(row["EventTime"])
                power_w = parse_decimal(row["AvPowerCons"])
                if power_w is None:
                    continue
                hourly_power_w[to_hour(ts)] += power_w
    return {hour: watts / 1000.0 for hour, watts in hourly_power_w.items()}


target_hours = load_target_hours(TARGETS_CSV)
hourly_load_kw = aggregate_hourly_load_from_release(REEFER_ZIP)

len(target_hours), len(hourly_load_kw)

(223, 8403)

In [4]:
def fallback_same_hour_last_week(hour: datetime, hourly_load_kw: dict[datetime, float]) -> float:
    week_ago = hour - timedelta(hours=168)
    day_ago = hour - timedelta(hours=24)
    if week_ago in hourly_load_kw and day_ago in hourly_load_kw:
        return 0.7 * hourly_load_kw[day_ago] + 0.3 * hourly_load_kw[week_ago]
    if day_ago in hourly_load_kw:
        return hourly_load_kw[day_ago]
    if week_ago in hourly_load_kw:
        return hourly_load_kw[week_ago]
    return 0.0


predictions = []
for hour in target_hours:
    pred_power_kw = fallback_same_hour_last_week(hour, hourly_load_kw)
    pred_p90_kw = pred_power_kw * 1.10
    predictions.append(
        {
            "timestamp_utc": iso_utc(hour),
            "pred_power_kw": round(pred_power_kw, 6),
            "pred_p90_kw": round(pred_p90_kw, 6),
        }
    )

predictions[:3]

[{'timestamp_utc': '2026-01-01T00:00:00Z',
  'pred_power_kw': 834.463337,
  'pred_p90_kw': 917.909671},
 {'timestamp_utc': '2026-01-01T01:00:00Z',
  'pred_power_kw': 826.299683,
  'pred_p90_kw': 908.929651},
 {'timestamp_utc': '2026-01-01T02:00:00Z',
  'pred_power_kw': 833.958823,
  'pred_p90_kw': 917.354705}]

In [5]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["timestamp_utc", "pred_power_kw", "pred_p90_kw"])
    writer.writeheader()
    writer.writerows(predictions)

OUTPUT_CSV

PosixPath('/Users/wegel/Documents/Projects/Hackathon 2026/participant_package/starter/output/predictions.csv')

## Next improvements

Good next steps for participants:
- add weather lags from `wetterdaten.zip`
- validate with forward-only time splits
- model peaks separately
- calibrate `pred_p90_kw` on a holdout period instead of using a fixed `10%` uplift

Organizer note: for final scoring, rerun the handed-in model/code on a hidden full target list that also includes the private timestamps.